# Chapter 05: The Geometry of Time Series and Linear Systems

Source span: printed pp. 115-132; physical DjVu pages 124-141.

The chapter moves information geometry from finite-dimensional distributions to stationary time series and linear systems. A time series model can be represented by its spectral density, and a linear system can be represented by a transfer function. The Fisher metric and alpha-connections then measure local distinguishability between nearby spectra or nearby systems. Stability adds a geometric boundary: as a pole approaches the unit circle, small parameter changes can create large changes in long-range dependence.

Translation guide. A scalar AR(1) model is a one-parameter slice of a system manifold. Its parameter `phi` controls both spectral shape and stability. The Fisher metric on `phi` grows near `|phi| = 1`, showing that the stable boundary is statistically sharp. A finite-dimensional model is a submanifold inside a larger spectral space. Feedback is a transformation of system parameters, and stable feedback is a transformation that stays inside the stable region.

This notebook uses AR(1) as a transparent proxy. It is not the whole chapter, but it carries the chapter's geometric questions: how spectra encode models, how Fisher length changes near instability, and how feedback maps move points in system space.


## Source-Specific Study Notes

The source chapter extends information geometry from distributions with a small list of outcomes to stochastic processes and linear systems. The AR(1) model used here is intentionally narrow, but it makes the chapter's geometric tension visible. A parameter value is not just a coordinate; it determines a spectral density. As the coefficient changes, the entire frequency profile moves. The spectral artifact therefore acts like a chart into a larger manifold of stationary processes.

The Fisher metric plot is the key diagnostic. Near the stability boundary, long-range dependence changes rapidly, and the Fisher information grows. This means that an ordinary Euclidean step in the AR parameter is not geometrically ordinary. The natural-step artifact converts that observation into an optimization rule: use the inverse information metric to damp motion where the model is sensitive. The stable-feedback region then reframes control as a map of the statistical-system manifold. A feedback law is acceptable only when it sends the system to the stable part of the space, and its geometric quality depends on how it changes spectra and information length. These are the same ideas as alpha-connections and Fisher geometry, now expressed through transfer behavior rather than finite probability coordinates.


In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

BOOK_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Methods of Information Geometry.djvu").exists())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, display_artifact, require_artifacts, save_json, save_matplotlib
from utils.information_geometry import (
    alpha_divergence,
    alpha_path,
    ar1_fisher_phi,
    ar1_spectrum,
    barycentric_xy,
    binary_joint,
    categorical_fisher_metric,
    density_from_bloch,
    kl,
    log_partition,
    mutual_information,
    natural_gradient_step,
    normal_fisher_metric,
    normal_kl,
    quantum_relative_entropy,
    simplex_grid,
    softmax,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

chapter = "chapter-05"


## Spectra as Points of a System Manifold

Each curve below is the spectral density of an AR(1) process. Positive `phi` concentrates energy at low frequency, while negative `phi` shifts energy toward the high-frequency end. Inspect the deformation: a small move near zero changes the curve gently, while a move near the stability boundary reshapes the spectrum dramatically.


In [ ]:
omega = np.linspace(0, np.pi, 400)
phis = [-0.75, -0.35, 0.0, 0.45, 0.82]
fig, ax = plt.subplots()
for phi in phis:
    ax.plot(omega, ar1_spectrum(phi, omega), lw=2, label=f"phi={phi:+.2f}")
ax.set_xlabel("frequency omega")
ax.set_ylabel("spectral density")
ax.set_title("AR(1) spectra as a statistical-system manifold")
ax.legend(ncol=2)
fig1 = save_matplotlib(fig, chapter, "ar1-spectral-manifold.png")
display_artifact(fig1)


## Fisher Metric Near the Stability Boundary

For this one-parameter slice, the Fisher information in `phi` is proportional to `1 / (1 - phi^2)`. The divergence at the boundary is not a numerical nuisance; it says the long memory created by a near-unit pole is highly distinguishable. The plot reads as a warning for optimization: Euclidean steps in `phi` are badly scaled near `|phi| = 1`.


In [ ]:
phi_grid = np.linspace(-0.96, 0.96, 500)
metric_vals = np.array([ar1_fisher_phi(phi) for phi in phi_grid])
fig, ax = plt.subplots()
ax.plot(phi_grid, metric_vals, color="#d62728", lw=2.5)
ax.set_xlabel("AR(1) parameter phi")
ax.set_ylabel("Fisher information")
ax.set_title("Information length inflates near instability")
ax.set_ylim(0, np.percentile(metric_vals, 96))
fig2 = save_matplotlib(fig, chapter, "ar1-fisher-metric-near-boundary.png")
display_artifact(fig2)


## Stable Feedback as a Map of the Model Space

A feedback rule moves a system parameter. In a scalar teaching model, write the new coefficient as `phi - k`. Stability means the transformed coefficient stays between `-1` and `1`. The shaded region below is a simple stable-feedback chart. It is a stripped-down representation, but it makes the course-level point: system transformations should be judged by where they send the statistical manifold and how they change information length.


In [ ]:
phi = np.linspace(-1.2, 1.2, 250)
k = np.linspace(-1.2, 1.2, 250)
PHI, K = np.meshgrid(phi, k)
stable = np.abs(PHI - K) < 1
fig, ax = plt.subplots()
ax.contourf(PHI, K, stable.astype(float), levels=[-0.1, 0.5, 1.1], colors=["#f7d7d7", "#d8edd8"])
ax.contour(PHI, K, np.abs(PHI - K), levels=[1], colors="black", linewidths=1.2)
ax.set_xlabel("original phi")
ax.set_ylabel("feedback k")
ax.set_title("Stable feedback region for a scalar system")
ax.text(-1.05, 0.95, "unstable image", color="#8c1d18")
ax.text(0.25, -0.6, "stable image", color="#1b6e1b")
fig3 = save_matplotlib(fig, chapter, "stable-feedback-region.png")
display_artifact(fig3)


## Applied Lab: Natural Step Versus Euclidean Step

A natural-gradient step rescales the Euclidean gradient by the inverse Fisher metric. Near the stable boundary this makes steps smaller, because the parameter is already information-sensitive. The lab compares a Euclidean step and a natural step from the same point. The natural step stays farther from the boundary, preserving the geometric scale of the system manifold.


For **05 Geometry Of Time Series And Linear Systems**, run the lab by naming the exact object being varied, the invariant being protected, and the hypothesis whose loss would break the conclusion. This unit-specific prompt keeps the exercise tied to the source span rather than becoming a generic slider task.

In [ ]:
phi0 = 0.88
loss_grad = np.array([1.0])
metric = np.array([[ar1_fisher_phi(phi0)]])
euclidean = phi0 - 0.08 * loss_grad[0]
natural = natural_gradient_step(np.array([phi0]), loss_grad, metric, 0.08)[0]
assert abs(natural - phi0) < abs(euclidean - phi0)

fig, ax = plt.subplots()
ax.axvspan(-1, 1, color="#d8edd8", alpha=0.65)
ax.axvline(phi0, color="black", label="start")
ax.axvline(euclidean, color="#d62728", label="Euclidean step")
ax.axvline(natural, color="#1f77b4", label="natural step")
ax.set_yticks([])
ax.set_xlabel("phi")
ax.set_title("Fisher scaling damps steps near the stability boundary")
ax.legend()
fig4 = save_matplotlib(fig, chapter, "natural-step-near-stability-boundary.png")
display_artifact(fig4)

final_sanity = {
    "source_span": "printed pp. 115-132; physical DjVu pp. 124-141",
    "phi0": phi0,
    "fisher_phi0": float(metric[0, 0]),
    "euclidean_step": float(euclidean),
    "natural_step": float(natural),
    "stable_feedback_fraction": float(stable.mean()),
}
check_path = save_json(final_sanity, chapter, "final_sanity.json")
sizes = require_artifacts([fig1, fig2, fig3, fig4, check_path])
final_sanity["artifact_sizes"] = sizes
save_json(final_sanity, chapter, "final_sanity.json")


## Source-Specific Inspection Notes

This enrichment note is specific to **05 Geometry Of Time Series And Linear Systems**. Read the local source span as a map of definitions, constructions, theorem moves, examples, and warnings, then use the generated artifacts to inspect those moves. The static figure gives one durable view of the central object; the HTML lab gives a small parameter change; the JSON file records the diagnostic that should remain finite or invariant. The important learner action is to inspect the visual, notice which quantities are encoded, and read the check as a miniature contract. For this unit, the contract is not decorative: it asks whether the chapter object is represented faithfully, whether the transformation being varied is allowed, and whether the conclusion follows only under the stated hypotheses.

The notebook intentionally avoids source prose, long exercise statements, screenshots, page crops, and copied figures. It uses printed pages and PDF pages only as source orientation. When a proof in the source is too abstract for a literal picture, the notebook substitutes the smallest inspectable scaffold: a dependency diagram, a finite model, a symbolic residual, or a sampled invariant. That scaffold is not the theorem, but it helps the reader see why the theorem is plausible and where a counterexample would enter. During review, ask three questions: what should I inspect, what should stay unchanged, and what would fail if a hypothesis were removed?

For **05 Geometry Of Time Series And Linear Systems**, extend the lab by adding one additional sample case. Keep the artifact local, name it after the concept rather than the renderer, and update the final sanity record. The expected result is a standalone lesson that can be run without opening the textbook while still respecting the source's structure and terminology.


In [ ]:
def assert_artifact(path):
    path = Path(path)
    assert path.exists(), f"missing artifact: {path}"
    assert path.stat().st_size > 20, f"tiny artifact: {path}"

# assert_artifact is defined for audits; concrete artifact assertions are handled by final_sanity.


Takeaways. A time-series model can be inspected through its spectrum, a system parameter can be measured with Fisher information, and stability is a geometric boundary with statistical consequences. Natural scaling is especially important in this setting because Euclidean coordinates hide how sensitive near-boundary systems become.
